# lb-whisper v2: fine-tune Whisper large-v3-turbo on FLEURS + open Luxembourgish data

Runtime > Change runtime type > GPU. The notebook only runs the scripts of the repository, in order: prepare, train, export, evaluate.

- The open data is read from the `shard-*.tar` archives written by `build_open_data.py`, in a Google Drive folder.
- Checkpoints, results and logs are written to Google Drive.
- **If the session is cut: reconnect and run all cells again.** Data preparation is redone (the VM disk is lost), training restarts from the last complete checkpoint on Drive.

How long it takes and which GPU to pick: `finetune/README.md`, section "v2 budget". The cell after data preparation prints the estimate for the data actually loaded; after a smoke test, the cell under the training cell prints it again from the speed measured on this GPU.

## 0. Settings
The only cell to edit.

In [ ]:
DRIVE_DIR = "/content/drive/MyDrive/lb-whisper"   # everything that must survive the session
SHARDS_DIR = f"{DRIVE_DIR}/open-data-v2"           # folder that holds shard-000.tar, shard-001.tar, ...
CONFIG = "turbo-lora-v2"                           # "turbo-lora-v2" or "turbo-full-v2" (see finetune/configs/)
TEXT_FIELD = "text"                                # "text" = verbatim transcripts, "text_normalised" = ZLS standard spelling
SMOKE_TEST = False                                 # True: 300 clips in a separate run folder, to time one step on this GPU
REPO = "https://github.com/Sitraka17/lb-whisper.git"
BRANCH = "v2-open-data"                            # "main" once the branch is merged

## 1. Drive, GPU, code

In [ ]:
import glob, json, math, os, shutil, subprocess
from google.colab import drive

drive.mount("/content/drive")
shards = sorted(glob.glob(f"{SHARDS_DIR}/shard-*.tar"))
assert shards, f"no shard-*.tar in {SHARDS_DIR}: upload the archives first, or fix SHARDS_DIR"
print(f"{len(shards)} archive(s), {sum(os.path.getsize(s) for s in shards) / 1e9:.2f} GB")
print(subprocess.run(["nvidia-smi", "-L"], capture_output=True, text=True).stdout or "no GPU: change the runtime type")
!df -h /content | tail -1

In [ ]:
%cd /content
if not os.path.isdir("lb-whisper"):
    !git clone -q --branch {BRANCH} {REPO}
%cd /content/lb-whisper/finetune
!git log -1 --format="code at %h (%cd)" --date=short
!pip install -q -r requirements.txt
!apt-get -qq install -y ffmpeg > /dev/null
# Colab ships torchao 0.10, too old for recent peft: it is removed.
!pip uninstall -y -q torchao

## 2. Hugging Face login (optional)
Only needed to publish a model; FLEURS and the base model download without login. The token comes from the Colab secret `HF_TOKEN` (key icon, notebook access on) and is never printed.

In [ ]:
from google.colab import userdata
from huggingface_hub import login

try:
    login(token=userdata.get("HF_TOKEN"))
    print("Hugging Face: logged in")
except Exception as error:
    print("Hugging Face: not logged in (fine unless you publish):", type(error).__name__)

## 3. Runs and results on Drive
`runs/` and `results/` of the repository become links to Drive folders, so checkpoints and scores outlive the VM.

In [ ]:
for name in ("runs", "results"):
    target = f"{DRIVE_DIR}/{name}"
    os.makedirs(target, exist_ok=True)
    if not os.path.islink(name):
        os.symlink(target, name)
    print(name, "->", os.path.realpath(name))
!df -h /content/drive | tail -1

## 4. Data
FLEURS `lb_lu` from the Hub plus the open data from the archives. FLEURS dev and test stay pure FLEURS; the open-data validation clips go to their own split, `validation_open`. Skipped when the dataset is already on the VM disk and was built with the current settings; rebuilt when `TEXT_FIELD`, `SHARDS_DIR` or the number of archives has changed since.

In [ ]:
DATA = "data/open_v2"
if os.path.exists(f"{DATA}/stats.json"):
    # A dataset left by an earlier run of this cell: keep it only if it was built with today's settings.
    built = (json.load(open(f"{DATA}/stats.json")).get("open_data") or [{}])[0]
    wanted = {"text_field": TEXT_FIELD, "folder": os.path.normpath(SHARDS_DIR), "archives": len(shards)}
    built = {key: os.path.normpath(built[key]) if key == "folder" and built.get(key) else built.get(key) for key in wanted}
    changed = {key: f"{built[key]} -> {value}" for key, value in wanted.items() if built[key] != value}
    if changed:
        print("settings changed since the dataset was built, rebuilding:", changed)
        shutil.rmtree(DATA)
if not os.path.exists(f"{DATA}/stats.json"):
    !python prepare_data.py --shards "{SHARDS_DIR}" --text-field {TEXT_FIELD} --out {DATA}
!du -sh {DATA}

In [ ]:
import yaml

stats = json.load(open(f"{DATA}/stats.json"))
cfg = yaml.safe_load(open(f"configs/{CONFIG}.yaml"))
for split in ("train", "validation", "validation_open", "test"):
    print(f"{split:16s} {stats[split]:6d} clips  {stats['hours'][split]:6.2f} h")
for report in stats.get("open_data", []):
    for name, entry in report["splits"]["train"]["by_source"].items():
        print(f"  train  {name:18s} {entry['clips']:6d} clips  {entry['hours']:6.2f} h  {entry['licence']}")


def estimate(smoke_metrics=None):
    """Hours for the full run. Whisper pads every clip to 30 s, so the cost follows the number of clips, not the hours.

    Without a smoke test: 1.0 s per clip, measured on v1 (turbo + LoRA, Colab T4: 2.1 h for 3 x 2,502 clips,
    with its three dev evaluations inside). The more frequent v2 evaluations are not in that figure.
    With a smoke test on this GPU: training and evaluation are timed apart. The smoke run evaluates 700 clips
    for 300 trained, the real run 700 for every 4,000 or 8,000, so its total time must not be scaled as it is."""
    train_cfg = cfg["train"]
    batch = train_cfg["per_device_train_batch_size"] * train_cfg["gradient_accumulation_steps"]
    epochs = train_cfg["num_train_epochs"]
    clip_passes = stats["train"] * epochs
    steps = math.ceil(stats["train"] / batch) * epochs
    evaluations = steps // train_cfg["eval_steps"]
    print(f"\n{CONFIG}: {clip_passes} clip passes = {steps} steps of {batch}, {evaluations} evaluations.")
    if smoke_metrics and os.path.exists(smoke_metrics):
        m = json.load(open(smoke_metrics))
        log = m["log_history"]
        eval_seconds = sum(v for row in log for k, v in row.items() if k.startswith("eval_") and k.endswith("_runtime"))
        rounds = sum(1 for row in log if "eval_fleurs_runtime" in row or "eval_runtime" in row)
        trained = m.get("train_clips", m["global_step"] * batch) * (m.get("epochs") or 1)
        per_clip = (m["train_runtime"] - eval_seconds) / trained
        per_evaluation = eval_seconds / rounds if rounds else 0.0
        train_hours, eval_hours = clip_passes * per_clip / 3600, evaluations * per_evaluation / 3600
        print(f"Smoke test on {m['gpu']}: {m['train_runtime']:.0f} s, of which {eval_seconds:.0f} s in {rounds} evaluation(s); "
              f"{per_clip:.2f} s per trained clip over {trained:.0f} clips, {per_evaluation:.0f} s per evaluation.")
        print(f"About {train_hours:.1f} h of training + {eval_hours:.1f} h of evaluation = {train_hours + eval_hours:.1f} h.")
        print("Still on the high side: a few hundred clips do not amortise the start of the data loader and of the GPU.")
    elif cfg.get("lora", {}).get("enabled"):
        print(f"About {clip_passes * 1.0 / 3600:.1f} h at 1.0 s per clip (v1 measurement on a T4, LoRA; evaluations of v2 not counted apart).")
    else:
        print("Full fine-tuning has never been timed: no estimate without a smoke test on this GPU (SMOKE_TEST = True).")
    print("An estimate, not a promise. Model loading and data preparation come on top at every session.")


estimate(f"runs/{CONFIG}-smoke/train_metrics.json")

## 5. Train
`--auto-resume` starts from the base model the first time and from the last complete checkpoint on Drive afterwards. The best checkpoint is the one with the lowest WER on FLEURS dev (`eval_fleurs_wer`); the open-data validation WER (`eval_open_wer`) is logged next to it and never mixed in.

With `SMOKE_TEST = True` the run uses 300 clips and its own folder (`runs/<config>-smoke`), so it can never be resumed by the real run. It trains one epoch of those 300 clips with one evaluation in the middle; the cell below the training cell then prints the estimate for the full run, with training and evaluation timed apart.

In [ ]:
if SMOKE_TEST:
    smoke_cfg = dict(cfg, run_name=f"{CONFIG}-smoke", output_dir=f"runs/{CONFIG}-smoke")
    smoke_cfg["train"] = dict(cfg["train"], num_train_epochs=1, eval_steps=10, save_steps=10, warmup_steps=2, logging_steps=5)
    yaml.safe_dump(smoke_cfg, open("/content/smoke.yaml", "w"), sort_keys=False)
    !python train.py --config /content/smoke.yaml --max-train-samples 300
else:
    !python train.py --config configs/{CONFIG}.yaml --auto-resume

In [ ]:
run = f"runs/{CONFIG}-smoke" if SMOKE_TEST else f"runs/{CONFIG}"
m = json.load(open(f"{run}/train_metrics.json"))
print("GPU:", m["gpu"], "| steps:", m["global_step"], "| best:", m["best_model_checkpoint"], m["metric_for_best_model"], m["best_metric"])
for row in m["log_history"]:
    if "eval_fleurs_wer" in row or "eval_open_wer" in row:
        print(row["step"], {k: round(v, 2) for k, v in row.items() if k.endswith("_wer")})
if SMOKE_TEST:
    estimate(f"{run}/train_metrics.json")   # the estimate again, now from the speed measured on this GPU

## 6. Export and evaluate
The reference number stays the full FLEURS test split (934 utterances) with the repository normaliser: `wer`. The result file also holds `wer_fleurs_norm` (FLEURS normalised references, for a comparison with BuzzASR) and `wer_whisper_basic`; `evaluate.py` explains each. The open-data validation split is scored separately, with a generation limit of 440 tokens instead of 225 because a few of its transcripts are longer than 225 tokens; the FLEURS test command is the same as for v1.

`export.py` writes `merged/` and `ct2-int8/` inside the run folder, so on Drive: see "Disk" in `finetune/README.md` for the sizes.

In [ ]:
assert not SMOKE_TEST, "set SMOKE_TEST = False and train the real run before exporting"
!python export.py --run runs/{CONFIG}
!python evaluate.py --model runs/{CONFIG}/merged --data {DATA} --split test
!python evaluate.py --model runs/{CONFIG}/merged --data {DATA} --split validation_open --max-new-tokens 440
!cat results/results.md

## If the session is cut
1. Reconnect, check that a GPU is attached, run all cells again. Nothing has to be edited.
2. Section 4 rebuilds the dataset on the new VM. Section 5 prints `resuming from runs/<config>/checkpoint-N` and continues; at most `save_steps` steps are lost.
3. A checkpoint that the cut left half-written (no `trainer_state.json`) is skipped; the one before is used.
4. If Drive is full, the save fails: free space or lower `save_total_limit` (never below 2, the best and the latest), then run again.